GraphRAG

Knowledge Graph из отчётов КТЖ и Матен Петролеум в Neo4j.

**Этапы:**
1. Развернуть Neo4j (Docker)
2. Извлечь сущности и связи из документов
3. Загрузить граф в Neo4j
4. Cypher-запросы для ответов на вопросы
5. Сравнение Vector RAG vs GraphRAG

## 1. Конфигурация и Neo4j

**Запуск Neo4j (Docker):**
```bash
docker run -d --name neo4j-rag -p 7474:7474 -p 7687:7687 -e NEO4J_AUTH=neo4j/password123 neo4j
```

Или Neo4j Aura Free: зарегистрируйтесь на https://neo4j.com/cloud/aura-free/ и получите URI + пароль.

In [17]:
from pathlib import Path
import os
from dotenv import load_dotenv

for p in [Path("."), Path("../assignment_1"), Path("..")]:
    if (p / ".env").exists():
        load_dotenv(p / ".env")
        break

DATA_DIR = Path("../assignment_1/data")
KTJ_PATH = DATA_DIR / "ktj_parsed.md"
MATNP_PATH = DATA_DIR / "matnp_parsed.md"

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password123")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

assert KTJ_PATH.exists(), f"Не найден {KTJ_PATH}. Запустите assignment_1."
print(f"Данные: {KTJ_PATH.parent}")
print(f"Neo4j: {NEO4J_URI}")

Данные: ..\assignment_1\data
Neo4j: bolt://localhost:7687


## 2. Извлечение сущностей и связей

LLM извлекает: Company, Project, Metric, Location и связи между ними.

In [20]:
from langchain_openai import ChatOpenAI
import json
import re

def extract_entities_llm(text: str, source: str, llm) -> tuple[list, list]:
    """Извлекает сущности и связи из фрагмента текста."""
    if not OPENAI_API_KEY:
        return [], []
    chunk = text[:3500] if len(text) > 3500 else text
    prompt = f"""Извлеки сущности (Company, Project, Metric, Location) и связи из текста годового отчёта.
Верни только JSON: {{"entities": [{{"type": "Company", "name": "..."}}], "relations": [{{"from": "A", "to": "B", "type": "HAS_PROJECT"}}]}}
Без markdown.

Текст ({source}):
{chunk}"""
    try:
        r = llm.invoke(prompt)
        out = r.content if hasattr(r, "content") else str(r)
        out = re.sub(r"^```(?:json)?\n?|\n?```$", "", out).strip()
        # Ищем JSON-объект в ответе (на случай лишнего текста)
        m = re.search(r"\{[\s\S]*\}", out)
        data = json.loads(m.group(0) if m else out) if out else {}
        entities = data.get("entities", []) or data.get("nodes", [])
        relations = data.get("relations", []) or data.get("relationships", [])
        for rel in relations:
            if "from" not in rel and "source" in rel:
                rel["from"] = rel.pop("source", "")
            if "to" not in rel and "target" in rel:
                rel["to"] = rel.pop("target", "")
        return entities, relations
    except Exception as e:
        return [], []

def load_docs() -> list[tuple[str, str]]:
    """Возвращает [(текст, source), ...]."""
    out = []
    if KTJ_PATH.exists():
        out.append((KTJ_PATH.read_text(encoding="utf-8"), "ktj"))
    if MATNP_PATH.exists():
        out.append((MATNP_PATH.read_text(encoding="utf-8"), "matnp"))
    return out

def extract_fallback(text: str, source: str) -> tuple[list, list]:
    """Fallback: извлечение по ключевым словам из отчётов КТЖ и Матен."""
    import re
    entities, relations = [], []
    patterns = [
        # Компании
        (r"КТЖ|ҚТЖ|Казахстан\s+темир\s+жолы|Қазақстан\s+темір\s+жолы|НК\s+«ҚТЖ»", "Company"),
        (r"Матен\s*Петролеум|Maten|МАТЕН ПЕТРОЛЕУМ", "Company"),
        (r"АО\s+«Кожан»|Кожан", "Company"),
        (r"КазМунайГаз|KazMunayGas", "Company"),
        (r"Chevron|ExxonMobil|Shell|Total|Eni|CNPC|Lukoil|Лукойл", "Company"),
        (r"Sino-Science|PwC|GRI|TCFD|SASB", "Company"),
        (r"Sozak\s*Oil|Priority\s*Oil|Арнаойл|СП Матин", "Company"),
        # Проекты
        (r"Достык\s*[–-]\s*Мойынты|Достык-Мойынты", "Project"),
        (r"Дарбаза\s*[–-]\s*Мактаарал|Дарбаза – Мактаарал", "Project"),
        (r"обход\s+Алматы|обход Алматы", "Project"),
        (r"Кара[-\s]?Арна|Кокарна|Восточная Кокарна", "Project"),
        (r"месторождени[ея]\s+Матин|Матин", "Project"),
        (r"месторождени[ея]\s+Морское|Морское", "Project"),
        (r"месторождени[ея]\s+Каратал|Каратал", "Project"),
        (r"Тенгиз|Теніз", "Project"),
        (r"КТК|Каспийский трубопровод", "Project"),
        # Метрики
        (r"ESG[-\s]?рейтинг|рейтинг\s+\d+|ESG", "Metric"),
        (r"EBITDA|NOPAT|LTIR", "Metric"),
        (r"млрд\s+тенге|млн\s+тонн", "Metric"),
        # Локации
        (r"Астана|Алматы|Актобе|Атырау|Караганда", "Location"),
        (r"Атырауская\s+область|Туркестанск\w*\s+область", "Location"),
    ]
    seen = set()
    for pat, etype in patterns:
        for m in re.finditer(pat, text, re.I):
            name = m.group(0).strip()
            if name not in seen and len(name) > 2:
                seen.add(name)
                entities.append({"type": etype, "name": name, "attrs": {}})
    # Связи: компания -> проект, компания -> локация
    comps = [e["name"] for e in entities if e["type"] == "Company"]
    projs = [e["name"] for e in entities if e["type"] == "Project"]
    locs = [e["name"] for e in entities if e["type"] == "Location"]
    for c in comps[:5]:
        for p in projs[:6]:
            relations.append({"from": c, "to": p, "type": "HAS_PROJECT"})
        for loc in locs[:2]:
            relations.append({"from": c, "to": loc, "type": "LOCATED_IN"})
    return entities, relations

print("Функции готовы.")

Функции готовы.


### Диагностика LLM

Запустите ячейку ниже, чтобы увидеть **сырой ответ LLM** и понять, почему извлечение даёт 0:
- API ключ не работает?
- LLM возвращает текст вместо JSON?
- JSON есть, но структура другая?
- Падает парсер (Exception)?

In [21]:
# ДИАГНОСТИКА: почему LLM возвращает 0?
# Запустите после ячейки с функциями и конфигом
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI

if not OPENAI_API_KEY:
    print("⚠ OPENAI_API_KEY не задан — LLM не вызывается")
else:
    splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
    text = KTJ_PATH.read_text(encoding="utf-8") if KTJ_PATH.exists() else ""
    chunk = splitter.split_text(text)[0] if text else ""
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY)
    
    print("Вызов LLM на 1 чанк...")
    try:
        r = llm.invoke(f"Извлеки сущности (Company, Project, Metric, Location) и связи. Верни JSON: {{\"entities\": [{{\"type\": \"Company\", \"name\": \"...\"}}], \"relations\": [{{\"from\": \"A\", \"to\": \"B\", \"type\": \"HAS_PROJECT\"}}]}}. Только JSON.\n\nТекст:\n{chunk[:1500]}")
        raw = r.content if hasattr(r, "content") else str(r)
        print("RAW ответ (первые 800 символов):")
        print(raw[:800])
        print("\n---")
        import json
        cleaned = re.sub(r"^```(?:json)?\n?|\n?```$", "", raw).strip()
        m = re.search(r"\{[\s\S]*\}", cleaned)
        data = json.loads(m.group(0) if m else cleaned)
        print("Распарсено: entities=", len(data.get("entities", [])), ", relations=", len(data.get("relations", [])))
    except Exception as e:
        print("ОШИБКА:", type(e).__name__, str(e))

Вызов LLM на 1 чанк...
RAW ответ (первые 800 символов):
```json
{
  "entities": [
    {
      "type": "Company",
      "name": "Национальная компания «Қазақстан темір жолы»"
    },
    {
      "type": "Project",
      "name": "Интегрированный годовой отчет"
    },
    {
      "type": "Metric",
      "name": "2024 год"
    },
    {
      "type": "Location",
      "name": "Казахстан"
    }
  ],
  "relations": [
    {
      "from": "Национальная компания «Қазақстан темір жолы»",
      "to": "Интегрированный годовой отчет",
      "type": "HAS_PROJECT"
    }
  ]
}
```

---
Распарсено: entities= 4 , relations= 1


In [22]:
# Разбивка на чанки (упрощённо) и извлечение
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None

all_entities = []
all_relations = []

for full_text, source in load_docs():
    chunks = splitter.split_text(full_text)
    for i, chunk in enumerate(chunks[:25]):  # 25 чанков на документ
        if llm:
            entities, relations = extract_entities_llm(chunk, source, llm)
            origin = "LLM" if (entities or relations) else None
        else:
            entities, relations = extract_fallback(chunk, source)
            origin = "fallback"
        if not entities and not relations and llm:
            entities, relations = extract_fallback(chunk, source)
            origin = "fallback"
        for e in entities:
            if isinstance(e, dict) and e.get("name"):
                e = dict(e)
                e["source"] = source
                e["type"] = e.get("type", "Entity")
                e["_origin"] = origin or "LLM"
                all_entities.append(e)
        for r in relations:
            if isinstance(r, dict) and r.get("from") and r.get("to"):
                r = dict(r)
                r["_origin"] = origin or "LLM"
                all_relations.append(r)
        if (i + 1) % 5 == 0:
            print(f"  {source}: {i+1}/{len(chunks)}")

# Дедупликация: одна сущность на (norm_name, type), объединяем source
def _norm(s):
    return " ".join(str(s).strip().lower().split()) if s else ""

seen_ent = {}  # (norm_name, type) -> entity
for e in all_entities:
    key = (_norm(e["name"]), e.get("type", "Entity"))
    e = dict(e)
    if key not in seen_ent:
        e["sources"] = [e.get("source", "")]
        seen_ent[key] = e
    else:
        ex = seen_ent[key]
        # Предпочитаем Title Case вместо ALL CAPS
        if e["name"] != e["name"].upper() and ex["name"] == ex["name"].upper():
            ex["name"] = e["name"]
        src = e.get("source", "")
        if src and src not in ex.get("sources", []):
            ex["sources"].append(src)
all_entities = list(seen_ent.values())
for e in all_entities:
    e["source"] = ", ".join(sorted(set(e.get("sources", [""])) - {""})) or "?"

# Нормализация связей: from/to -> каноническое имя
norm_to_canonical = {_norm(e["name"]): e["name"] for e in all_entities}
seen_rel = set()
clean_relations = []
for r in all_relations:
    cf = norm_to_canonical.get(_norm(r["from"]), r["from"])
    ct = norm_to_canonical.get(_norm(r["to"]), r["to"])
    key = (cf, r.get("type"), ct)
    if key not in seen_rel:
        seen_rel.add(key)
        r = dict(r)
        r["from"], r["to"] = cf, ct
        clean_relations.append(r)
all_relations = clean_relations

# Если связи пусты, но есть сущности — создаём эвристические связи
if not all_relations and all_entities:
    comps = [e["name"] for e in all_entities if e.get("type") in ("Company", "Organization")]
    projs = [e["name"] for e in all_entities if e.get("type") == "Project"]
    for e in all_entities:
        n = e["name"]
        if ("КТЖ" in n or "Матен" in n) and n not in comps:
            comps.append(n)
        if ("Достык" in n or "Дарбаза" in n or "обход" in n) and n not in projs:
            projs.append(n)
    if comps and projs:
        for c in list(set(comps))[:5]:
            for p in list(set(projs))[:6]:
                all_relations.append({"from": c, "to": p, "type": "HAS_PROJECT", "_origin": "heuristic"})
    locs = [e["name"] for e in all_entities if e.get("type") == "Location"]
    if comps and locs and len(all_relations) < 5:
        for c in list(set(comps))[:3]:
            for loc in list(set(locs))[:3]:
                all_relations.append({"from": c, "to": loc, "type": "LOCATED_IN", "_origin": "heuristic"})
    if not all_relations:
        names = list({e["name"] for e in all_entities})[:10]
        for i in range(min(len(names) - 1, 8)):
            all_relations.append({"from": names[i], "to": names[i + 1], "type": "RELATED_TO", "_origin": "heuristic"})

print(f"Извлечено сущностей: {len(all_entities)}, связей: {len(all_relations)}")

  ktj: 5/47
  ktj: 10/47
  ktj: 15/47
  ktj: 20/47
  ktj: 25/47
  matnp: 5/30
  matnp: 10/30
  matnp: 15/30
  matnp: 20/30
  matnp: 25/30
Извлечено сущностей: 270, связей: 181


In [24]:
# Вывод извлечённых сущностей и связей (для проверки качества)
from collections import Counter
llm_ent = sum(1 for e in all_entities if e.get("_origin") == "LLM")
fb_ent = sum(1 for e in all_entities if e.get("_origin") == "fallback")
llm_rel = sum(1 for r in all_relations if r.get("_origin") == "LLM")
heur_rel = sum(1 for r in all_relations if r.get("_origin") == "heuristic")
print("Источник: LLM —", llm_ent, "сущ.,", llm_rel, "связ. | fallback —", fb_ent, "сущ. | heuristic —", heur_rel, "связ.\n")

print("=" * 60)
print("СУЩНОСТИ (по типам)")
print("=" * 60)
for etype in sorted(set(e.get("type", "?") for e in all_entities)):
    items = [e for e in all_entities if e.get("type") == etype]
    print(f"\n[{etype}] ({len(items)} шт.)")
    for e in items[:15]:
        orig = e.get("_origin", "?")
        print(f"  • {e['name']}  [source: {e.get('source')}, от: {orig}]")
    if len(items) > 15:
        print(f"  ... и ещё {len(items) - 15}")

print("\n" + "=" * 60)
print("СВЯЗИ")
print("=" * 60)
for r in all_relations:
    orig = r.get("_origin", "?")
    print(f"  {r.get('from')} --[{r.get('type')}]--> {r.get('to')}  [от: {orig}]")

Источник: LLM — 270 сущ., 181 связ. | fallback — 0 сущ. | heuristic — 0 связ.

СУЩНОСТИ (по типам)

[Company] (95 шт.)
  • Национальная компания «Қазақстан темір жолы»  [source: ktj, от: LLM]
  • АО «НК «ҚТЖ»  [source: ktj, от: LLM]
  • Компания  [source: ktj, от: LLM]
  • PwC  [source: ktj, от: LLM]
  • Штадлер  [source: ktj, от: LLM]
  • АО «Самрук-Қазына»  [source: ktj, от: LLM]
  • АО «Фонд национального благосостояния «Самрук-Қазына»  [source: ktj, от: LLM]
  • АО «KTZ Express»  [source: ktj, от: LLM]
  • ООО «Китайско-казахстанская международная логистическая компания Ляньюньган»  [source: ktj, от: LLM]
  • ООО «CRK TERMINAL»  [source: ktj, от: LLM]
  • филиал «Xinjiang KTZ International Logisnics Co.LTD»  [source: ktj, от: LLM]
  • Компания с иностранным капиталом «Xinjiang KTZ International Logistics Co.LTD»  [source: ktj, от: LLM]
  • Общество с ограниченной ответственностью «KTZ Express Hong Kong»  [source: ktj, от: LLM]
  • СП «YuXinOu (Chongjing) Logistics Co. Ltd»  [source

## 3. Загрузка в Neo4j

In [25]:
from neo4j import GraphDatabase

def load_to_neo4j(entities: list, relations: list):
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    with driver.session() as session:
        session.run("MATCH (n) DETACH DELETE n") #сначала удаляются все узлы и связи в Neo4j -> добавляются новые
        for e in entities:
            if e.get("type") and e.get("name"):
                label = e["type"].replace(" ", "_")
                attrs = e.get("attrs", {})
                attrs["source"] = e.get("source", "")
                props = ", ".join(f"n.{k} = ${k}" for k in attrs)
                session.run(f"MERGE (n:{label} {{name: $name}}) SET {props}", {"name": e["name"], **attrs})
        for r in relations:
            if r.get("from") and r.get("to") and r.get("type"):
                rel_type = r["type"].replace(" ", "_")
                session.run(f"""
                    MATCH (a) WHERE a.name = $f
                    MATCH (b) WHERE b.name = $t
                    MERGE (a)-[r:{rel_type}]->(b)
                """, f=r["from"], t=r["to"])
    driver.close()

try:
    load_to_neo4j(all_entities, all_relations)
    print("Граф загружен в Neo4j.")
except Exception as e:
    print(f"Ошибка: {e}. Убедитесь, что Neo4j запущен.")

Граф загружен в Neo4j.


## 4. Cypher-запросы

In [29]:
from neo4j import GraphDatabase

def run_cypher(query: str) -> list:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    with driver.session() as session:
        result = session.run(query)
        records = list(result)
    driver.close()
    return records

def _get_name(node):
    if node is None: return "?"
    if hasattr(node, "get"): return node.get("name", "?")
    try: return node["name"] if "name" in node else "?"
    except: return "?"

def pretty(records):
    """Удобный вывод: 'from--[rel]-->to' для записей с p,r,n."""
    if not records:
        print("  (нет результатов)")
        return
    r0 = records[0]
    keys = list(r0.keys())
    if "p" in keys and "r" in keys and "n" in keys:
        for rec in records:
            p, r, n = rec["p"], rec["r"], rec["n"]
            rel = getattr(r, "type", "?")
            print(f"  {_get_name(p)} --[{rel}]--> {_get_name(n)}")
    else:
        for rec in records:
            print("  " + " | ".join(str(rec[k]) for k in keys))

# 1. Проекты Достык–Мойынты
print("=" * 60)
print("1. Какие инфраструктурные проекты связаны с направлением Достык – Мойынты?")
print("=" * 60)
q1 = """MATCH (p:Project)-[r]-(n) WHERE p.name CONTAINS 'Достык' OR p.name CONTAINS 'Мойынты' RETURN p, r, n LIMIT 20"""
pretty(run_cypher(q1))

# 2. Компании в обоих отчётах (source содержит и ktj, и matnp)
print("\n" + "=" * 60)
print("2. Какие компании упоминаются в обоих отчётах?")
print("=" * 60)
q2 = """MATCH (c:Company) WHERE c.source IS NOT NULL AND c.source CONTAINS 'ktj' AND c.source CONTAINS 'matnp' RETURN c.name AS name, c.source AS source"""
both = run_cypher(q2)
if both:
    for r in both[:20]:
        print(f"  • {r['name']} (источники: {r['source']})")
    if len(both) > 20:
        print(f"  ... и ещё {len(both)-20}")
else:
    print("  Нет компаний с source 'ktj, matnp' (разные отчёты → разные компании).")
    print("\n  Примеры компаний по отчётам:")
    ktj = run_cypher("MATCH (c:Company) WHERE c.source CONTAINS 'ktj' RETURN c.name AS name LIMIT 5")
    matnp = run_cypher("MATCH (c:Company) WHERE c.source CONTAINS 'matnp' RETURN c.name AS name LIMIT 5")
    print("  КТЖ:", ", ".join(r["name"] for r in ktj) if ktj else "—")
    print("  Матен:", ", ".join(r["name"] for r in matnp) if matnp else "—")

# 3. ESG-рейтинг КТЖ и экологические проекты
print("\n" + "=" * 60)
print("3. Связь между ESG-рейтингом КТЖ и экологическими проектами")
print("=" * 60)
q3 = """MATCH (c)-[r]-(m) WHERE (c:Company OR c:Metric) AND (m.name CONTAINS 'ESG' OR m.name CONTAINS 'эколог' OR m.name CONTAINS 'выброс' OR m.name CONTAINS 'природоохран') AND (c.name CONTAINS 'ҚТЖ' OR c.name CONTAINS 'КТЖ') RETURN c, r, m LIMIT 15"""
recs3 = run_cypher(q3)
if recs3:
    for rec in recs3:
        c, r, m = rec["c"], rec["r"], rec["m"]
        rel = getattr(r, "type", "?")
        print(f"  {_get_name(c)} --[{rel}]--> {_get_name(m)}")
else:
    q3b = """MATCH (c:Company)-[r:HAS_METRIC]-(m:Metric) WHERE c.name CONTAINS 'ҚТЖ' AND m.name CONTAINS 'ESG' RETURN c.name, m.name"""
    for r in run_cypher(q3b):
        print(f"  {r['c.name']} --[HAS_METRIC]--> {r['m.name']}")

1. Какие инфраструктурные проекты связаны с направлением Достык – Мойынты?
  строительство вторых путей Достык – Мойынты --[HAS_PROJECT]--> АО «НК «ҚТЖ»

2. Какие компании упоминаются в обоих отчётах?
  Нет компаний с source 'ktj, matnp' (разные отчёты → разные компании).

  Примеры компаний по отчётам:
  КТЖ: Национальная компания «Қазақстан темір жолы», АО «НК «ҚТЖ», Компания, PwC, Штадлер
  Матен: АО «Матен Петролеум», АО «Кожан», ТОО «Арнаойл», ТОО «СП «Матин», АО «ОрдабасыМунайГаз»

3. Связь между ESG-рейтингом КТЖ и экологическими проектами
  АО «НК «ҚТЖ» --[HAS_METRIC]--> ESG-рейтинг


## 5. Сравнение Vector RAG vs GraphRAG

| Вопрос | Vector RAG | GraphRAG | Лучше |
|--------|------------|----------|-------|
| Какие проекты связаны с Достык–Мойынты? | Поиск по эмбеддингам, может пропустить контекст | Прямой Cypher по связям Project–Company | **GraphRAG** |
| Какие компании в обоих отчётах? | Нужно искать в двух индексах и пересекать | Один запрос: `source` содержит ktj и matnp | **GraphRAG** |
| Связь ESG-рейтинга КТЖ с экопроектами? | Семантический поиск по чанкам | Граф связей Company–Metric–Project | **GraphRAG** |
| Точные числа (доход 2024, млн тонн)? | Чанк с таблицей — ближайший контекст | Метрики есть как узлы, но без значений | **Vector RAG** |
| Длинные цитаты, контекст абзаца? | Возвращает исходный текст | Только имена сущностей и типы связей | **Vector RAG** |

### Вывод

- **GraphRAG** — для вопросов о связях (кто с кем связан, кросс-отчётные сущности, путь в графе).
- **Vector RAG** — для фактоидных вопросов, точных чисел, длинного контекста из исходного текста.
- **Гибрид** — объединение обоих подходов даёт наиболее полные ответы.